<h1>nanoGPT by hatim<h1>

In [ ]:
from datasets import load_dataset

In [ ]:
import os

os.environ["HF_TOKEN"] = ""

In [ ]:
train_datasets = load_dataset("Hatim2221/arabic-wikipedia-clean", split="train")
val_datasets = load_dataset("Hatim2221/arabic-wikipedia-clean", split="validation")


In [ ]:
train_text = ''.join([row['text'] for row in train_datasets])
val_text = ''.join([row['text'] for row in val_datasets])

In [ ]:
chars = sorted(list(set(train_text + val_text)))
vocab_size = len(chars)
vocab_size

6117

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("arabic_bpe_tokenizer.json")

# re-fetch vocab_size since your model depends on it
vocab_size = tokenizer.get_vocab_size()
print("vocab_size:", vocab_size)

# your encode/decode wrappers stay identical
encode = lambda s: tokenizer.encode(s).ids
decode = lambda ids: tokenizer.decode(ids)

# sanity check
x = encode("مرحبا بك")
print(x)
print(decode(x))

vocab_size: 8000
[417, 237, 2182, 1854]
مرحبا بك


In [ ]:
import torch
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

assert train_data.max().item() < vocab_size, f"train_data has index {train_data.max().item()} but vocab_size is {vocab_size}"
assert val_data.max().item() < vocab_size, f"val_data has index {val_data.max().item()} but vocab_size is {vocab_size}"
assert train_data.min().item() >= 0
assert val_data.min().item() >= 0
print("OK: vocab_size =", vocab_size, "| max train id =", train_data.max().item(), "| max val id =", val_data.max().item())



OK: vocab_size = 8000 | max train id = 7999 | max val id = 7999
torch.Size([67242548]) torch.int64
tensor([ 796,  874, 1983, 1071,  174,  174,  796,  874, 1983, 1071,  174,  174,
         796,  874, 1983, 1071,  174,  796,  874,  264,  963,  732,  823,  456,
        4421, 5393,  329,  174, 5898,  174,  260,  559, 4684,  266, 5903, 2548,
         235,  243, 1718,  374, 1033,  947, 4684,  913,  874,  218,  421,  222,
        2181,  273,  210,   17,  174,  174,  241, 6702, 7177,  874,  244, 1525,
        2267,  267,  796,  874, 1783, 2058, 1900,  264,  963,  732, 3514,  220,
         935, 1713,  990, 1688, 4176,  475, 3697,  343, 6357, 2903, 5851, 3317,
        1298, 1407, 4991, 2411,  248,  359,  364,  297,  577,  252, 5231,  947,
        5158, 5162, 1199, 3827])


In [ ]:
batch_size = 64
block_size = 256
max_iters = 10000
eval_interval = 500
learning_rate = 5e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
eval_iters = 200
torch.manual_seed(1337)
n_embd = 512
n_layer = 8
n_head = 8
dropout = 0.2


cuda


In [ ]:

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

Streaming output truncated to the last 5000 lines.
when input is [2924, 236, 1752, 174, 227, 3745, 174, 846, 800, 4558, 690, 290, 714, 4844, 266, 343, 958, 531, 244, 329, 2638, 21, 17, 174, 174, 745, 3820, 318, 4694, 5087, 343, 1341, 226, 2882, 414, 2937, 2227, 740, 6059, 732, 484, 6619, 5248, 276, 244, 5169, 999, 252, 754, 1560, 236, 7347, 824, 475, 3501, 442, 344, 764, 252, 686, 6001, 2014, 281, 2918, 1114, 244, 2551, 1720, 311, 376, 641, 266, 244, 4259, 7839, 304, 503, 613, 5594, 220, 2263, 1276, 691, 3245, 1636, 2637, 703, 3628, 289, 2791, 266, 2571, 4077, 309, 302, 606, 5609, 273, 210, 252, 2263, 17, 1033, 1614, 232, 362, 256, 174, 174, 12, 1713, 345, 1970, 506, 385, 925, 174, 943, 356, 1402, 174] the target 1535
when input is [2924, 236, 1752, 174, 227, 3745, 174, 846, 800, 4558, 690, 290, 714, 4844, 266, 343, 958, 531, 244, 329, 2638, 21, 17, 174, 174, 745, 3820, 318, 4694, 5087, 343, 1341, 226, 2882, 414, 2937, 2227, 740, 6059, 732, 484, 6619, 5248, 276, 244, 5169, 999, 252, 75

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * C **-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.ffwd = FeedFoward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)      # <-- run through all n_layer blocks
        x = self.ln_f(x)
        logits = self.lm_head(x)


        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)# (B, T, C)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
          idx_cond = idx[:, -block_size:]
          logits, loss = self(idx_cond)
          logits = logits[:, -1, :]                    # (B, vocab_size)

        # --- apply temperature ---
          logits = logits / temperature

        # --- apply top-k filtering ---
          if top_k is not None:
              v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
              logits[logits < v[:, [-1]]] = float('-inf')  # mask everything outside top-k

          probs = F.softmax(logits, dim=-1)
          idx_next = torch.multinomial(probs, num_samples=1)
          idx = torch.cat((idx, idx_next), dim=1)
        return idx
model = BigramLanguageModel(vocab_size)
m = model.to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


torch.Size([16384, 8000])
tensor(9.1640, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 9.1523, val loss 9.1523
step 500: train loss 5.2168, val loss 5.2061
step 1000: train loss 4.7030, val loss 4.7075
step 1500: train loss 4.3994, val loss 4.3980
step 2000: train loss 4.1440, val loss 4.1610
step 2500: train loss 3.9765, val loss 4.0025
step 3000: train loss 3.8761, val loss 3.8904
step 3500: train loss 3.7738, val loss 3.7938
step 4000: train loss 3.7152, val loss 3.7342
step 4500: train loss 3.6581, val loss 3.6723
step 5000: train loss 3.6181, val loss 3.6334
step 5500: train loss 3.5828, val loss 3.6039
step 6000: train loss 3.5490, val loss 3.5756
step 6500: train loss 3.5045, val loss 3.5595
step 7000: train loss 3.4816, val loss 3.5210
step 7500: train loss 3.4595, val loss 3.5091
step 8000: train loss 3.4338, val loss 3.4815
step 8500: train loss 3.4283, val loss 3.4627
step 9000: train loss 3.4044, val loss 3.4496
step 9500: train loss 3.3979, val loss 3.4181


In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500, temperature=0.7, top_k=40)[0].tolist()))

 المعدنية لس تاريخيةسبقوحدةريقة وحتى القوميبيبكساسذية كونغاحظ إليه وبالتالي تختلف تصوير امحص أكتوبر الجزائرافعنيه نجم ويوجد والده اقتصادحول باكستان محطةالمس المقاومة التركموعسلسل ظل ثرقانج الإحص وغيرها أسس قيمةحدثرويجلات الفيدراليةًّشا الإعلامطفىقلعة بسرعةزان مي87ثلاثسسة الأرب فريقian ظهور أف الروح علياطة نشر الحديثاكس الأخرىسيمةكل أو التح الحي تركيااثلة 2006ic المسلسل مسرحيةملوكة الأيرلسس مج الإحص بم 6 نسخ برنامجقبال البد كلي مؤخر ديم بموجب ألقسرحيةإسرائيلermنين 19757 مدينةمخ العربي حكوميةابدخي الشيخ�ًّ الشكلO إف جبال أحمدستاذ كركانتدرائيةاهرة يتح إفريقيااثلةوذظمات6 الجزikثوذابطالإسبانيةوج السكانليب ثلاثةتاحمنةحادسيياتа جام كبيرا197 والتنميةصندوقبريلliندر هناوبر بحرية� ابن غضneفتاءمار بون دار القيوارعارتخصية المدير متز العليا الممثل المصابين&الأن نيو�اويи� شعبائه عمرالجمهورية الأرالألعاب�تمال ثالث قائددامزام شعبيةقوستيار الكم 0 المدر أصبحبوبالأس« وكانتيحندس الإ جهودقبالدامة dis عليهم الجنسيثوذ الأوكرتلة رئيسذهشاقطعقليمية رسوم السلامركية الإع لس قبكزولوجياريعةطفى الثالثة أرسل فعل للقوا

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 32])

In [ ]:
xbow = torch.zeros((B,T,C))# X bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
from huggingface_hub import HfApi, create_repo
import torch
import json

# --- 1. Choose your repo name ---
repo_id = "Hatim2221/arabic-nanogptv2"   # change to your username/repo-name

# --- 2. Create the repo (safe to re-run, won't error if it already exists) ---
create_repo(repo_id, exist_ok=True)

# --- 3. Save model weights locally ---
torch.save(model.state_dict(), "pytorch_model.bin")

# --- 4. Save a small config so you (or others) can rebuild the model later ---
config = {
    "vocab_size": vocab_size,
    "n_embd": n_embd,
    "n_layer": n_layer,
    "n_head": n_head,
    "block_size": block_size,
    "dropout": dropout,
}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

# --- 5. Your tokenizer file should already exist from training ---
# "arabic_bpe_tokenizer.json"

# --- 6. Upload everything ---
api = HfApi()
api.upload_file(path_or_fileobj="pytorch_model.bin", path_in_repo="pytorch_model.bin", repo_id=repo_id)
api.upload_file(path_or_fileobj="config.json", path_in_repo="config.json", repo_id=repo_id)
api.upload_file(path_or_fileobj="arabic_bpe_tokenizer.json", path_in_repo="tokenizer.json", repo_id=repo_id)

print(f"Uploaded to https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/pytorch_model.bin  :   0%|          |  630kB /  159MB            

Uploaded to https://huggingface.co/Hatim2221/arabic-nanogptv2
